# Probe: pycolmap dense API signatures (env-spec deps on GPU)
Confirms undistort_images / patch_match_stereo / stereo_fusion signatures + options fields, and that photogrammetry_gpu_env5 installs via the environment spec (not %pip) on Serverless GPU env5.

In [ ]:
import json
import pycolmap
out = {"pycolmap": getattr(pycolmap, "__version__", "?"), "has_cuda": bool(getattr(pycolmap, "has_cuda", None))}
# Full docstrings — pybind11 embeds the exact C++ signature(s) + defaults.
for fn in ("undistort_images", "patch_match_stereo", "stereo_fusion"):
    f = getattr(pycolmap, fn, None)
    out[fn + "_doc"] = (getattr(f, "__doc__", "") or "").strip()
# Options defaults (repr each non-callable attr).
for cls in ("StereoFusionOptions", "PatchMatchOptions", "UndistortCameraOptions"):
    c = getattr(pycolmap, cls, None)
    if c is None:
        out[cls] = "MISSING"; continue
    try:
        inst = c()
        fields = {}
        for a in dir(inst):
            if a.startswith("_"):
                continue
            try:
                v = getattr(inst, a)
                if not callable(v):
                    fields[a] = repr(v)
            except Exception as e:  # noqa: BLE001
                fields[a] = f"<err {type(e).__name__}>"
        out[cls] = fields
    except Exception as e:  # noqa: BLE001
        out[cls] = f"instantiate-err: {type(e).__name__}: {e}"
print(json.dumps(out, indent=2))
dbutils.notebook.exit(json.dumps(out))